# YOLOv8 Weed Detection Training
This notebook handles the dataset preparation and training for **YOLOv8**.

### ⚠️ Cache Fix Included
This version automatically cleans up corrupted `.cache` files to prevent 'NpzFile' or 'PermissionError' issues on Windows.

In [1]:
import os
import json
import shutil
import random
from pathlib import Path
from ultralytics import YOLO

# Settings
DATASET_ROOT = r"c:\Users\ahmad\Desktop\computer vision\cottonweed"
WORK_DIR = r"c:\Users\ahmad\Desktop\computer vision\weed_yolo_benchmark"
FRACTION = 0.10  
EPOCHS = 50
BATCH_SIZE = 8
IMG_SIZE = 640
DEVICE = 0

CLASS_NAMES = ["weed1", "weed2", "weed3", "weed4", "weed5", "weed6", "weed7", "weed8", "weed9", "weed10", "weed11", "weed12"]

os.makedirs(WORK_DIR, exist_ok=True)

### 1. Prepare Dataset Subset

In [2]:
def create_subset(fraction):
    subset_name = f"subset_{int(fraction * 100)}"
    subset_root = Path(WORK_DIR) / "datasets" / subset_name
    
    if (subset_root / "SUBSET_READY.json").exists():
        return subset_root

    print(f"Creating {subset_name} subset...")
    os.makedirs(subset_root / "images" / "train", exist_ok=True)
    os.makedirs(subset_root / "labels" / "train", exist_ok=True)
    
    for split in ["val", "test"]:
        src_img = Path(DATASET_ROOT) / "images" / split
        if src_img.exists():
            shutil.copytree(src_img, subset_root / "images" / split, dirs_exist_ok=True)
            shutil.copytree(Path(DATASET_ROOT) / "labels" / split, subset_root / "labels" / split, dirs_exist_ok=True)

    train_images = list((Path(DATASET_ROOT) / "images" / "train").glob("*.jpg"))
    random.seed(42)
    selected = random.sample(train_images, int(len(train_images) * fraction))
    
    for img_path in selected:
        shutil.copy2(img_path, subset_root / "images" / "train")
        lbl_path = Path(DATASET_ROOT) / "labels" / "train" / f"{img_path.stem}.txt"
        if lbl_path.exists():
            shutil.copy2(lbl_path, subset_root / "labels" / "train")

    yaml_content = f"train: { (subset_root / 'images' / 'train').as_posix() }\nval: { (subset_root / 'images' / 'val').as_posix() }\nnc: {len(CLASS_NAMES)}\nnames: {CLASS_NAMES}"
    with open(subset_root / "data.yaml", "w") as f:
        f.write(yaml_content)
    
    with open(subset_root / "SUBSET_READY.json", "w") as f:
        json.dump({"fraction": fraction}, f)
    return subset_root

subset_path = create_subset(FRACTION)

### 2. Training (Real-time output)

In [3]:
exp_name = f"yolov8_data{int(FRACTION*100)}_aug"
exp_dir = Path(WORK_DIR) / "runs" / exp_name
data_yaml = subset_path / "data.yaml"
best_ckpt = exp_dir / "weights" / "best.pt"

def clear_cache(path):
    """Force delete .cache files to prevent permission errors on Windows."""
    print("Cleaning up dataset cache files...")
    for cache in path.rglob("*.cache"):
        try:
            os.remove(cache)
            print(f"  - Removed {cache.name}")
        except Exception as e:
            print(f"  - Warning: Could not remove {cache.name} (it may be locked): {e}")

if best_ckpt.exists():
    print("Weights found. Skipping training.")
else:
    # Clear cache to avoid corrupted NpzFile errors
    clear_cache(subset_path)
    
    model = YOLO("yolov8n.pt")
    model.train(
        data=str(data_yaml),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        project=str(Path(WORK_DIR) / "runs"),
        name=exp_name,
        exist_ok=True,
        workers=0
    )

Cleaning up dataset cache files...
  - Removed train.cache
  - Removed val.cache
Ultralytics 8.4.48  Python-3.11.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\ahmad\Desktop\computer vision\weed_yolo_benchmark\datasets\subset_10\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momen

### 3. Validation & Results

In [4]:
model = YOLO(str(exp_dir / "weights" / "best.pt"))
results = model.val(data=str(data_yaml), imgsz=IMG_SIZE)

summary = {
    "model": "yolov8",
    "fraction": FRACTION,
    "map50": float(results.box.map50),
    "precision": float(results.box.mp),
    "recall": float(results.box.mr)
}
with open(exp_dir / "EXPERIMENT_DONE.json", "w") as f:
    json.dump(summary, f)

Ultralytics 8.4.48  Python-3.11.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
Model summary (fused): 73 layers, 3,007,988 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1681.4630.9 MB/s, size: 146.7 KB)
val: Scanning C:\Users\ahmad\Desktop\computer vision\weed_yolo_benchmark\datasets\subset_10\labels\val.cache... 1342 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1342/1342  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 84/84 10.4it/s 8.0s0.2s
                   all       1342       2004      0.903      0.824      0.906      0.855
                 weed1        359        501      0.947      0.859      0.933      0.872
                 weed2        240        280      0.951      0.895      0.947      0.867
                 weed3        203        264      0.897      0.759      0.897        0.8
                 weed4         78         93      0.947      0.882   